# Bài 1.

In [2]:
import pandas as pd
import numpy as np

from collections import Counter

from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
df_train = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab5/UIT-ViOCD/train.json').T
df_dev = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab5/UIT-ViOCD/dev.json').T
df_test = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab5/UIT-ViOCD/test.json').T

In [4]:
df_train.head()

,review,label,domain
0,gói hàng cẩn thận . chơi pubg với lie...,non-complaint,mobile
1,mình góp ý thật nhé . . đừng bắt pha...,complaint,app
2,"máy khá đẹp , pin trâu vân tay nhạy nhạ...",complaint,mobile
3,một mớ lỗi : không xem lại được bà...,complaint,app
4,đặc mẫu xanh mà sao giao mẫu này vậy...,complaint,fashion


In [5]:
X_train = df_train["review"].astype(str).tolist()
y_train = df_train["domain"].tolist()

In [6]:
X_dev = df_dev["review"].astype(str).tolist()
y_dev = df_dev["domain"].tolist()

In [7]:
X_test = df_test["review"].astype(str).tolist()
y_test = df_test["domain"].tolist()

In [8]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
num_classes = len(label_encoder.classes_)

In [9]:
y_dev = label_encoder.transform(y_dev)
y_test = label_encoder.transform(y_test)

In [10]:
VOCAB_SIZE = 30000
MAX_LEN = 128
BATCH_SIZE = 64

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="[UNK]",
    filters=""
)

tokenizer.fit_on_texts(X_train)

In [11]:
def encode_texts(texts, max_len=MAX_LEN):
    ids = tokenizer.texts_to_sequences(texts)
    ids = pad_sequences(
        ids,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    mask = (ids != 0).astype("int32")
    return ids, mask

In [12]:
train_ids, train_mask = encode_texts(X_train)
dev_ids, dev_mask     = encode_texts(X_dev)
test_ids, test_mask   = encode_texts(X_test)

In [13]:
def make_dataset(input_ids, labels=None, shuffle=False):
    if labels is not None:
        ds = tf.data.Dataset.from_tensor_slices((input_ids, labels))
    else:
        ds = tf.data.Dataset.from_tensor_slices(input_ids)

    if shuffle:
        ds = ds.shuffle(10000)

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [14]:
train_ds = make_dataset(
    train_ids, y_train, shuffle=True
)

dev_ds = make_dataset(
    dev_ids, y_dev
)

test_ds = make_dataset(
    test_ids
)

## Xây dựng model

In [15]:
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()

        pos = np.arange(max_len)[:, None]
        i = np.arange(d_model)[None, :]

        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        angle_rads = pos * angle_rates

        pe = np.zeros((max_len, d_model))
        pe[:, 0::2] = np.sin(angle_rads[:, 0::2])
        pe[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pe = tf.constant(pe[None, ...], dtype=tf.float32)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

In [16]:
class EncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(d_ff, activation="relu"),
            layers.Dense(d_model)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, mask=None, training=False):
        attn = self.mha(
            x, x, x,
            attention_mask=mask,
            training=training
        )
        x = self.norm1(x + self.dropout1(attn, training=training))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out, training=training))

        return x


In [17]:
class TransformerEncoder(tf.keras.Model):
    def __init__(
        self,
        vocab_size,
        max_len,
        num_classes,
        num_layers=4,
        d_model=256,
        num_heads=8,
        d_ff=512,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = layers.Embedding(
            vocab_size, d_model, mask_zero=True
        )
        self.pos_encoding = PositionalEncoding(max_len, d_model)

        self.encoders = [
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ]

        self.dropout = layers.Dropout(dropout)
        self.pool = layers.GlobalAveragePooling1D()
        self.classifier = layers.Dense(num_classes)

    def call(self, x, training=False):
        mask = self.embedding.compute_mask(x)
        if mask is not None:
            mask = mask[:, None, None, :]  # (B,1,1,L)

        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)

        for encoder in self.encoders:
            x = encoder(x, mask=mask, training=training)

        x = self.pool(x)
        return self.classifier(x)

In [18]:
model = TransformerEncoder(
    vocab_size=30000,
    max_len=128,
    num_classes=num_classes,
    num_layers=3,
    d_model=256,
    num_heads=8,
    d_ff=512
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=3e-4,
        beta_1=0.9,
        beta_2=0.98,
        epsilon=1e-9
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.build((None, 128))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'transformer_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "transformer_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ ?                      │   0 (unbuilt) │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_block (EncoderBlock)    │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_block_1 (EncoderBlock)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_block_2 (EncoderBlock)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.fit(
    train_ds,
    validation_data=dev_ds,
    epochs=10
)


Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'positional_encoding' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


69/69 ━━━━━━━━━━━━━━━━━━━━ 39s 217ms/step - accuracy: 0.5079 - loss: 1.2617 - val_accuracy: 0.5639 - val_loss: 1.0379
Epoch 2/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.5879 - loss: 0.9889 - val_accuracy: 0.6405 - val_loss: 0.9015
Epoch 3/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.7235 - loss: 0.6648 - val_accuracy: 0.7828 - val_loss: 0.5604
Epoch 4/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.8203 - loss: 0.5025 - val_accuracy: 0.8303 - val_loss: 0.4452
Epoch 5/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.8441 - loss: 0.4105 - val_accuracy: 0.8467 - val_loss: 0.3919
Epoch 6/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.8585 - loss: 0.3726 - val_accuracy: 0.8631 - val_loss: 0.3884
Epoch 7/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.8597 - loss: 0.3681 - val_accuracy: 0.8558 - val_loss: 0.3943
Epoch 8/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.8802 - loss: 0.3220 - val_accuracy: 0.8595 - val_loss: 

In [20]:
logits = model.predict(test_ds)
y_pred = np.argmax(logits, axis=-1)

9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 187ms/step


In [21]:
from sklearn.metrics import accuracy_score
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.9089253187613844


In [22]:
from sklearn.metrics import classification_report, f1_score

print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

Macro F1: 0.8686015637555287


# Bài 2:

In [23]:
df_train2 = pd.read_json("/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab5/PhoNER/word/train_word.json", lines=True)
df_dev2 = pd.read_json("/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab5/PhoNER/word/dev_word.json", lines=True)
df_test2 = pd.read_json("/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab5/PhoNER/word/test_word.json", lines=True)

In [24]:
def build_vocab(seqs):
  counter = Counter()
  for s in seqs:
    for w in s:
      counter[w] += 1

  vocab = {"<PAD>":0, "<UNK>":1}
  for w, _ in counter.items():
    vocab[w] = len(vocab)
  return vocab

word_vocab = build_vocab(df_train2["words"])
tag_vocab = build_vocab(df_train2["tags"])

idx2tag = {i: t for t, i in tag_vocab.items()}

In [25]:
MAX_LEN = 120
def encode_words(words):
  return [word_vocab.get(w, 1) for w in words]

def encode_tags(tags):
  if tags is None:
    return []
  return [tag_vocab[t] for t in tags]

def pad(seq):
  if len(seq) >= MAX_LEN:
    return seq[:MAX_LEN]
  return seq + [0]*(MAX_LEN - len(seq))

In [26]:
X_train = np.array([pad(encode_words(x)) for x in df_train2["words"]])
y_train = np.array([pad(encode_tags(x)) for x in df_train2["tags"]])

X_dev = np.array([pad(encode_words(x)) for x in df_dev2["words"]])
y_dev = np.array([pad(encode_tags(x)) for x in df_dev2["tags"]])

X_test = np.array([pad(encode_words(x)) for x in df_test2["words"]])
y_test = np.array([pad(encode_tags(x)) for x in df_test2["tags"]])

In [27]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()

        pos = np.arange(max_len)[:, None]
        i = np.arange(d_model)[None, :]

        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / d_model)
        angles = pos * angle_rates

        pe = np.zeros((max_len, d_model))
        pe[:, 0::2] = np.sin(angles[:, 0::2])
        pe[:, 1::2] = np.cos(angles[:, 1::2])

        self.pe = tf.constant(pe[None, ...], dtype=tf.float32)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

In [28]:
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.Wq = tf.keras.layers.Dense(d_model)
        self.Wk = tf.keras.layers.Dense(d_model)
        self.Wv = tf.keras.layers.Dense(d_model)
        self.Wo = tf.keras.layers.Dense(d_model)

    def split_heads(self, x):
        B = tf.shape(x)[0]
        L = tf.shape(x)[1]

        x = tf.reshape(x, (B, L, self.num_heads, self.d_head))
        return tf.transpose(x, [0, 2, 1, 3])

    def call(self, q, k, v, mask=None):
        Q = self.split_heads(self.Wq(q))
        K = self.split_heads(self.Wk(k))
        V = self.split_heads(self.Wv(v))

        scores = tf.matmul(Q, K, transpose_b=True)
        scores /= tf.math.sqrt(tf.cast(self.d_head, tf.float32))

        if mask is not None:
            scores += (1.0 - mask) * -1e9

        attn = tf.nn.softmax(scores, axis=-1)
        out = tf.matmul(attn, V)

        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, (tf.shape(out)[0], tf.shape(out)[1], self.d_model))

        return self.Wo(out)

In [29]:
class FeedForward(tf.keras.layers.Layer):
    def __init__(self, d_model, dff):
        super().__init__()
        self.fc1 = tf.keras.layers.Dense(dff, activation="relu")
        self.fc2 = tf.keras.layers.Dense(d_model)

    def call(self, x):
        return self.fc2(self.fc1(x))

In [30]:
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout=0.1):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, dff)

        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, x, mask=None, training=False):
        attn_out = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_out, training=training))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out, training=training))

        return x

In [31]:
class TransformerNER(tf.keras.Model):
    def __init__(
        self,
        vocab_size,
        max_len,
        num_layers,
        d_model,
        num_heads,
        dff,
        num_tags,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = tf.keras.layers.Embedding(
            vocab_size, d_model, mask_zero=True
        )
        self.pos_encoding = PositionalEncoding(max_len, d_model)

        self.enc_layers = [
            EncoderBlock(d_model, num_heads, dff, dropout)
            for _ in range(num_layers)
        ]

        self.dropout = tf.keras.layers.Dropout(dropout)
        self.classifier = tf.keras.layers.Dense(num_tags)

    def call(self, x, training=False):
        mask = tf.cast(x != 0, tf.float32)
        attn_mask = mask[:, tf.newaxis, tf.newaxis, :]

        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)

        for layer in self.enc_layers:
            x = layer(x, mask=attn_mask, training=training)

        return self.classifier(x)

In [32]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

def masked_loss(y_true, y_pred):
    loss = loss_fn(y_true, y_pred)
    mask = tf.cast(y_true != 0, tf.float32)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_accuracy(y_true, y_pred):
    y_pred = tf.argmax(y_pred, axis=-1)
    mask = tf.cast(y_true != 0, tf.float32)

    correct = tf.cast(y_true == y_pred, tf.float32)
    correct = correct * mask

    return tf.reduce_sum(correct) / tf.reduce_sum(mask)

In [33]:
BATCH_SIZE = 64

train_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

dev_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_dev, y_dev))
    .batch(BATCH_SIZE)
)

In [34]:
VOCAB_SIZE = int(
    max(
        X_train.max(),
        X_dev.max(),
        X_test.max()
    )
) + 1

In [35]:
NUM_TAGS = int(
    max(
        y_train.max(),
        y_dev.max(),
        y_test.max()
    )
) + 1

In [48]:
MAX_LEN = X_train.shape[1]

model = TransformerNER(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    num_layers=3,
    d_model=256,
    num_heads=8,
    dff=512,
    num_tags=NUM_TAGS
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=masked_loss,
    metrics=[masked_accuracy]
)

In [37]:
model.summary()

Model: "transformer_ner"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_1           │ ?                      │   0 (unbuilt) │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_block_3 (EncoderBlock)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_block_4 (EncoderBlock)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_block_5 (EncoderBlock)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [49]:
model.fit(
    train_ds,
    validation_data=dev_ds,
    epochs=30
)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'positional_encoding_2' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


79/79 ━━━━━━━━━━━━━━━━━━━━ 33s 198ms/step - loss: 1.3837 - masked_accuracy: 0.7301 - val_loss: 1.1297 - val_masked_accuracy: 0.7547
Epoch 2/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - loss: 1.0018 - masked_accuracy: 0.7913 - val_loss: 1.1011 - val_masked_accuracy: 0.7547
Epoch 3/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - loss: 0.9754 - masked_accuracy: 0.7894 - val_loss: 0.9959 - val_masked_accuracy: 0.7547
Epoch 4/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - loss: 0.8332 - masked_accuracy: 0.7951 - val_loss: 0.7015 - val_masked_accuracy: 0.7987
Epoch 5/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - loss: 0.6128 - masked_accuracy: 0.8274 - val_loss: 0.5900 - val_masked_accuracy: 0.8306
Epoch 6/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - loss: 0.5036 - masked_accuracy: 0.8512 - val_loss: 0.4927 - val_masked_accuracy: 0.8588
Epoch 7/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - loss: 0.4194 - masked_accuracy: 0.8737 - val_loss: 0.4339 - val_masked_accuracy: 0.8721
Epoch 8/30
79/79 ━━━━━

In [50]:
y_pred_2 = model.predict(X_test)

94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step


In [51]:
pred_ids = np.argmax(y_pred_2, axis=-1)
y_true = []
y_pred = []

for true_seq, pred_seq in zip(y_test, pred_ids):
  t, p = [], []
  for ti, pi in zip(true_seq, pred_seq):
    if ti == 0:
      continue
    t.append(idx2tag[ti])
    p.append(idx2tag[pi])
  y_true.append(t)
  y_pred.append(p)

In [52]:
from seqeval.metrics import f1_score

print("F1:", f1_score(y_true, y_pred))

F1: 0.7970643454514423
